<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:15px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#8B0000; overflow:hidden"><b> Multimodal Fake News Detection System </b></div>

## Multimodal Fake News Detection System

## Overview

A deep learning system that combines BERT (text analysis) and ResNet (image analysis) to detect fake news articles with high accuracy.
## Architecture

Text Branch: BERT-base-uncased (768-dim embeddings)
Image Branch: ResNet-18 (512-dim features)
Fusion: Multi-layer perceptron combining both modalities
Output: Binary classification (Real/Fake)


## if you find this helpful please consider upvoting.

In [1]:
"""
Multimodal Fake News Detection System
Combines BERT for text analysis and ResNet for image analysis
Binary classification: Real (0) vs Fake (1) news
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizer
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import random
import os



def set_seed(seed=42):
    """
    Set seeds for reproducibility across libraries
    WHY: Ensures consistent results across runs for debugging and comparison
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



class MultimodalNewsDataset(Dataset):
    """
    Dataset class that loads both text and images
    WHY: PyTorch requires custom Dataset class for multimodal data
    """
    
    def __init__(self, dataframe, tokenizer, max_length=128, transform=None):
        """
        Args:
            dataframe: pandas DataFrame with 'text', 'image_path', 'label'
            tokenizer: BERT tokenizer for text processing
            max_length: maximum sequence length for BERT
            transform: image transformations
        """
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get text and tokenize
        # WHY: BERT requires specific tokenization format with special tokens
        text = str(self.data.loc[idx, 'text'])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        # Load and transform image
        # WHY: ResNet expects normalized images of specific size
        image_path = self.data.loc[idx, 'image_path']
        try:
            image = Image.open(image_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
        except Exception as e:
            # WHY: Handle corrupted images gracefully with a blank tensor
            print(f"Error loading image {image_path}: {e}")
            image = torch.zeros(3, 224, 224)
        
        # Get label
        label = torch.tensor(self.data.loc[idx, 'label'], dtype=torch.long)
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'image': image,
            'label': label
        }



2026-01-27 16:17:48.616503: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769530668.829279      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769530668.896851      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769530669.474438      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769530669.474474      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769530669.474477      55 computation_placer.cc:177] computation placer alr

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> MODEL ARCHITECTURE</b></div>

In [2]:

class MultimodalFakeNewsDetector(nn.Module):

    
    def __init__(self, bert_model_name='bert-base-uncased', num_classes=2, dropout=0.3):
        super(MultimodalFakeNewsDetector, self).__init__()
        

        self.bert = BertModel.from_pretrained(bert_model_name)
        self.bert_hidden_size = self.bert.config.hidden_size  # 768 for base
        

        self.resnet = models.resnet18(pretrained=True)
        # Remove final classification layer
        self.resnet_feature_size = self.resnet.fc.in_features  # 512
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-1])
        

        self.fusion = nn.Sequential(
            nn.Linear(self.bert_hidden_size + self.resnet_feature_size, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, input_ids, attention_mask, image):
  
        bert_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_features = bert_output.pooler_output  # [batch_size, 768]
        

        image_features = self.resnet(image)  # [batch_size, 512, 1, 1]
        image_features = image_features.view(image_features.size(0), -1)  # [batch_size, 512]
        
     
        combined_features = torch.cat([text_features, image_features], dim=1)
        
        # Final classification
        output = self.fusion(combined_features)
        
        return output

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b>TRAINING</b></div>

In [3]:

def train_epoch(model, dataloader, criterion, optimizer, device):

    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    progress_bar = tqdm(dataloader, desc='Training')
    
    for batch in progress_bar:
       
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        image = batch['image'].to(device)
        labels = batch['label'].to(device)
        
       
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, image)
        loss = criterion(outputs, labels)
        
       
        loss.backward()
      
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        progress_bar.set_postfix({'loss': loss.item(), 'acc': 100 * correct / total})
    
    return total_loss / len(dataloader), 100 * correct / total


def evaluate(model, dataloader, criterion, device):

    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image = batch['image'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids, attention_mask, image)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(dataloader), all_preds, all_labels



<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> VISUALIZATION</b></div>

In [4]:

def plot_confusion_matrix(y_true, y_pred, save_path='confusion_matrix.png'):

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Real', 'Fake'],
                yticklabels=['Real', 'Fake'])
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.title('Confusion Matrix - Fake News Detection')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_training_history(train_losses, val_losses, train_accs, val_accs, save_path='training_history.png'):
    """
    Plot training curves
    WHY: Diagnose overfitting and training dynamics
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss plot
    ax1.plot(train_losses, label='Train Loss', marker='o')
    ax1.plot(val_losses, label='Val Loss', marker='s')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)
    
    # Accuracy plot
    ax2.plot(train_accs, label='Train Acc', marker='o')
    ax2.plot(val_accs, label='Val Acc', marker='s')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Training history saved to {save_path}")


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b>INFERENCE</b></div>

In [5]:

def predict_single_sample(model, text, image_path, tokenizer, transform, device):

    model.eval()
    
    # Tokenize text
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    
    # Load and transform image
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0)  # Add batch dimension
    
    # Move to device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    image = image.to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(input_ids, attention_mask, image)
        probabilities = torch.softmax(outputs, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0][predicted_class].item()
    
    return {
        'prediction': 'Fake' if predicted_class == 1 else 'Real',
        'confidence': confidence,
        'probabilities': {
            'Real': probabilities[0][0].item(),
            'Fake': probabilities[0][1].item()
        }
    }


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> MAIN </b></div>

In [6]:

def main():
    """
    Main training pipeline
    """

    set_seed(42)
    

    config = {
        'data_path': 'fake_news_data.csv',
        'batch_size': 16,
        'num_epochs': 10,
        'learning_rate': 2e-5,
        'max_length': 128,
        'num_workers': 4,
        'save_dir': 'models',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu'
    }
    
    print(f"Using device: {config['device']}")
    os.makedirs(config['save_dir'], exist_ok=True)
    

    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    

    print("Loading data...")
    
 
    df = create_synthetic_data(num_samples=1000)
    
    train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
    
    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    

    train_dataset = MultimodalNewsDataset(train_df, tokenizer, config['max_length'], transform)
    val_dataset = MultimodalNewsDataset(val_df, tokenizer, config['max_length'], transform)
    test_dataset = MultimodalNewsDataset(test_df, tokenizer, config['max_length'], transform)
    
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], 
                             shuffle=True, num_workers=config['num_workers'])
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], 
                           shuffle=False, num_workers=config['num_workers'])
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], 
                            shuffle=False, num_workers=config['num_workers'])
    

    print("Initializing model...")
    model = MultimodalFakeNewsDetector().to(config['device'])
    
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config['learning_rate'])
    

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    )
    

    print("Starting training")
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    best_val_loss = float('inf')
    
    for epoch in range(config['num_epochs']):
        print(f"\nEpoch {epoch + 1}/{config['num_epochs']}")
        

        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, config['device']
        )
        

        val_loss, val_preds, val_labels = evaluate(
            model, val_loader, criterion, config['device']
        )
        

        val_f1 = f1_score(val_labels, val_preds, average='weighted')
        val_acc = 100 * np.mean(np.array(val_preds) == np.array(val_labels))
        
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, Val F1: {val_f1:.4f}")
        

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        

        scheduler.step(val_loss)
        

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, os.path.join(config['save_dir'], 'best_model.pt'))
            print(" Saved best model")
    

    plot_training_history(train_losses, val_losses, train_accs, val_accs)
    

    print("\nLoading best model for testing...")
    checkpoint = torch.load(os.path.join(config['save_dir'], 'best_model.pt'))
    model.load_state_dict(checkpoint['model_state_dict'])
    

    print("\nEvaluating on test set...")
    test_loss, test_preds, test_labels = evaluate(
        model, test_loader, criterion, config['device']
    )
    

    test_acc = 100 * np.mean(np.array(test_preds) == np.array(test_labels))
    test_f1 = f1_score(test_labels, test_preds, average='weighted')
    
    print(f"\nTest Results:")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    print(f"Test F1-Score: {test_f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(test_labels, test_preds, 
                               target_names=['Real', 'Fake'], digits=4))
    
    
    plot_confusion_matrix(test_labels, test_preds)
    
    print("\n✓ Training complete!")


def create_synthetic_data(num_samples=1000):
    """
    Create synthetic dataset for demonstration
    WHY: Allows testing without external data dependencies
    
    In production, replace with real data:
    df = pd.read_csv('fake_news_data.csv')
    """
    import os
    from PIL import Image
    
  
    os.makedirs('synthetic_images', exist_ok=True)
    
    data = {
        'text': [],
        'image_path': [],
        'label': []
    }
    
    real_texts = [
        "Breaking news: Scientists discover new treatment for common disease",
        "Government announces new infrastructure plan worth billions",
        "Local community celebrates annual cultural festival",
        "Technology company releases quarterly earnings report",
        "Sports team wins championship after decades of effort"
    ]
    
    fake_texts = [
        "SHOCKING: Celebrity caught in unbelievable scandal you won't believe",
        "This one weird trick will change your life forever doctors hate it",
        "Secret government conspiracy revealed by anonymous source",
        "Miracle cure discovered that big pharma doesn't want you to know",
        "Aliens confirmed to be living among us says random website"
    ]
    
    for i in range(num_samples):
        # WHY: Balanced dataset prevents biased model
        label = i % 2
        
        if label == 0:  
            text = random.choice(real_texts)
        else:  
            text = random.choice(fake_texts)
        
        
        img = Image.new('RGB', (224, 224), 
                       color=(random.randint(0, 255), 
                             random.randint(0, 255), 
                             random.randint(0, 255)))
        image_path = f'synthetic_images/img_{i}.jpg'
        img.save(image_path)
        
        data['text'].append(text)
        data['image_path'].append(image_path)
        data['label'].append(label)
    
    return pd.DataFrame(data)


if __name__ == "__main__":
    main()


Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Train Loss: 0.0003, Train Acc: 100.00%
Val Loss: 0.0002, Val Acc: 100.00%, Val F1: 1.0000
✓ Saved best model

Epoch 10/10


Evaluating: 100%|██████████| 10/10 [00:01<00:00,  6.20it/s]


Train Loss: 0.0003, Train Acc: 100.00%
Val Loss: 0.0001, Val Acc: 100.00%, Val F1: 1.0000
✓ Saved best model
Training history saved to training_history.png

Loading best model for testing...

Evaluating on test set...


Evaluating: 100%|██████████| 10/10 [00:01<00:00,  6.16it/s]



Test Results:
Test Loss: 0.0001
Test Accuracy: 100.00%
Test F1-Score: 1.0000

Classification Report:
              precision    recall  f1-score   support

        Real     1.0000    1.0000    1.0000        75
        Fake     1.0000    1.0000    1.0000        75

    accuracy                         1.0000       150
   macro avg     1.0000    1.0000    1.0000       150
weighted avg     1.0000    1.0000    1.0000       150

Confusion matrix saved to confusion_matrix.png

✓ Training complete!
